# Convolutional Neural Networks

Import of the used packages and methods

In [65]:
import numpy as np
import tensorflow as tf
import tsfel
from tensorflow.keras import layers, models
import pandas as pd
import datetime

The Tensorboard extension was used to visualize the history of training

In [66]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


# Split Train/Test in 5 folds
splitted to Dicts of:
- meta_tr (train)
- meta_te (test)

and each group will appear exactly once in the test set across all folds

In [67]:
from sklearn.model_selection import GroupKFold
from auswertung_features.build_features import split_into_label_blocks, load_session

meta = pd.read_csv("../auswertung_features/meta.csv")
gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
meta_tr: dict[int, pd.DataFrame] = {}
meta_te: dict[int, pd.DataFrame] = {}

dict_blocks = {}
meta_blocks = {"subject_id":[], "hand": [], "block_id":[]}
for i, r in meta.iterrows():
        csv_path = r["rel_path"]
        df_session = load_session(csv_path)

        #Error Handling für leere dfs
        if df_session is None:
            continue

        blocks = split_into_label_blocks(df_session)
        for block in blocks:
            meta_blocks["subject_id"].append(r["subject_id"])
            meta_blocks["hand"].append(r["hand"])
            meta_blocks["block_id"].append(len(dict_blocks))
            dict_blocks[len(dict_blocks)] = block
meta_blocks = pd.DataFrame(meta_blocks)

for fold, (tr_idx, te_idx) in enumerate(gkf.split(meta_blocks, groups=meta_blocks["subject_id"])):
    meta_tr[fold] = meta_blocks.iloc[tr_idx].reset_index(drop=True)
    meta_te[fold] = meta_blocks.iloc[te_idx].reset_index(drop=True)


[WARN] ../ArduinoApps/emg_messung/python/messdaten/messung_l_170.csv: is empty


## Convert dataset to match TensorFlow Conventions

Input needs to be in shape (batch, steps, features). My own blocks (ndarray) are in the shape of (sample_length, sensor_channels). The steps are the sequence length which should be learned.

In [68]:
def build_xy_from_blocks(meta_df, dict_blocks, df_features: pd.DataFrame | None = None):
    """
    this function turns the block to the x time series and y as the label.
    Alternatively it can be used to input another dataframe with features which are not the same length. This is done to get a multi input in the later ANNs
    :param meta_df: this holds the meta dataframe which has the information of which time series is supposed to be used
    :param dict_blocks: the dictionary which is used as a look up table for the blocks. The meta_df has the index
    :param df_features: the dataframe with extra information. This only works correctly because we are using the whole sequence which where build and put in the dataframe sequentially. The "block_id" correlates with the index. It will break if it not done with the same meta.csv or the meta.csv wis shuffled
    :return: X_List, np.asarray(y_list) or X_List, F_List, np.asarray(y_list)
    """
    X_list, F_list, y_list = [], [], []
    for _, row in meta_df.iterrows():
        Xb, yb = dict_blocks[int(row["block_id"])]
        X_list.append(np.asarray(Xb, dtype=np.float32))
        y_list.append(int(yb))   # bleibt 0..4
        if df_features is not None:
            fb = df_features.iloc[int(row["block_id"]),6:] # the 6 is bc I am reusing function where meta information is in the same dataframe. We will not use this metainformation here
            F_list.append(np.asarray(fb, dtype=np.float32))
    if df_features is not None:
        return X_list, F_list, np.asarray(y_list, dtype=np.int32)
    else:
        return X_list, np.asarray(y_list, dtype=np.int32)


In [69]:
def make_dataset(X_list, y, F_list = None, batch_size=32, training=True):
    X_list = [np.asarray(x, dtype=np.float32) for x in X_list]
    y = np.asarray(y, dtype=np.int32)
    if F_list is not None:
        F_list = np.asarray(F_list, dtype=np.float32)
        F = F_list.shape[1]
        def gen():
            for x_i, f_i, y_i in zip(X_list, F_list, y):
                yield(x_i, f_i), y_i
        output_signature = (
        (
            tf.TensorSpec(shape=(None, 4), dtype=tf.float32),
            tf.TensorSpec(shape=(F,), dtype=tf.float32),
        ),
        tf.TensorSpec(shape=(), dtype=tf.int32),
        )
    else:
        def gen():
            for x_i, y_i in zip(X_list, y):
                yield x_i, y_i
        output_signature = (
        tf.TensorSpec(shape=(None, 4), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32),
        )

    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)

    if training:
        ds = ds.shuffle(buffer_size=len(y), reshuffle_each_iteration=True)

    if F_list is not None:
        ds = ds.padded_batch(
        batch_size=batch_size,
        padded_shapes=(([None, 4], [F]), []),
        padding_values=((0.0, 0.0), 0),
    ).prefetch(tf.data.AUTOTUNE)
    else:
        ds = ds.padded_batch(
            batch_size=batch_size,
            padded_shapes=([None, 4], []),
            padding_values=(0.0, 0)
        ).prefetch(tf.data.AUTOTUNE)

    return ds

In [70]:
def seq_to_windows(x,y, win_len=256, stride=128, tsfel_cfg: dict | None = None, fs: int = 500):
    """
    :param x: (T, 4) float32 Time Series
    :param y: int (Klasse)
    :param win_len: int (window length)
    :param stride: int (stride or step is the same here)
    :param tsfel_cfg: tsfel config dict. can be get by tsfel.get_features_by_domain()
    :param fs: int, is the frequency with which the features should be build. Without tsfel it is not used
    returns: Xw (n_windows, win_len, 4), yw (n_windows,)
    """
    T = x.shape[0]
    if T < win_len:
        return np.empty((0, win_len, x.shape[1]), dtype=np.float32), np.empty((0,), dtype=np.int32)

    starts = range(0, T - win_len + 1, stride)
    Xw = np.stack([x[s:s+win_len] for s in starts]).astype(np.float32)
    yw = np.full((Xw.shape[0],), int(y), dtype=np.int32)
    if tsfel_cfg is not None:
        feats_df = tsfel.time_series_features_extractor(tsfel_cfg, Xw, fs = fs)
        Fw = np.asarray(feats_df, dtype=np.float32)
        return Xw, Fw, yw
    return Xw, yw

In [71]:
def build_windowed_xy(meta_df, dict_blocks, win_len=256, stride=128, tsfel_cfg: dict | None = None, fs: int = 500):
    """
    builds windowed parts without features
    """
    X_all = []
    y_all = []
    F_all = []

    for _, row in meta_df.iterrows():
        Xb, yb = dict_blocks[int(row["block_id"])]
        Xb = np.asarray(Xb, dtype=np.float32)
        yb = int(yb)  # 0..4

        if tsfel_cfg is not None:
            Xw, Fw, yw = seq_to_windows(Xb, yb, win_len=win_len, stride=stride, tsfel_cfg=tsfel_cfg, fs=fs)
            F_all.append(Fw)
        else:
            Xw, yw = seq_to_windows(Xb, yb, win_len=win_len, stride=stride)

        if len(yw) == 0:
            continue

        X_all.append(Xw)
        y_all.append(yw)

    X_all = np.concatenate(X_all, axis=0)  # (N, win_len, 4)
    y_all = np.concatenate(y_all, axis=0)  # (N,)
    if tsfel_cfg is not None:
        F_all = np.concatenate(F_all, axis=0)
        return X_all, F_all, y_all
    return X_all, y_all


# Shaping the Batch
This defines how the dataset should be formed. This one gives the whole sequence as batches to train

In [72]:
fold = 1
str_hand = "r"
multi_input = True # This can be used to get a feature table in the dataset

mask_tr = meta_tr[fold]["hand"]==str_hand
mask_te = meta_te[fold]["hand"]==str_hand

if multi_input:
    df_features = tsfel_feature()
    X_tr, F_tr, y_tr = build_xy_from_blocks(meta_tr[fold][mask_tr], dict_blocks, df_features=df_features)
    X_te, F_te, y_te = build_xy_from_blocks(meta_te[fold][mask_te], dict_blocks, df_features=df_features)

    ds_tr = make_dataset(X_tr, y_tr, F_tr, batch_size=32, training=True).repeat(2)
    ds_te = make_dataset(X_te, y_te, F_te, batch_size=32, training=False).repeat(2)
    # Shape-Check
    for (xb, fb), yb in ds_tr.take(1):
        print("xb:", xb.shape, "fb", fb.shape, "yb:", yb.shape)  # xb z.B. (32, 1190, 4)
else:
    X_tr, y_tr = build_xy_from_blocks(meta_tr[fold][mask_tr], dict_blocks)
    X_te, y_te = build_xy_from_blocks(meta_te[fold][mask_te], dict_blocks)

    ds_tr = make_dataset(X_tr, y_tr, batch_size=32, training=True)
    ds_te = make_dataset(X_te, y_te, batch_size=32, training=False)
    for xb, yb in ds_tr.take(1):
        print("xb:", xb.shape, "yb:", yb.shape)  # xb z.B. (32, 1190, 4)


xb: (32, 1622, 4) fb (32, 624) yb: (32,)


This part defines the sequences into windows, which also makes every window the same length. The ends will be cut off

In [ ]:
fold = 0
win_len, stride = 250, 50 # stride is like a step here
str_hand = "r"
multi_input = False

mask_tr = meta_tr[fold]["hand"]==str_hand
mask_te = meta_te[fold]["hand"]==str_hand
if multi_input:
    tsfel_cfg = tsfel.get_features_by_domain()
    X_tr_w, F_tr_w, y_tr_w = build_windowed_xy(meta_tr[fold][mask_tr], dict_blocks, win_len, stride, tsfel_cfg=tsfel_cfg)
    X_te_w, F_te_w, y_te_w = build_windowed_xy(meta_tr[fold][mask_tr], dict_blocks, win_len, stride, tsfel_cfg=tsfel_cfg)

    print(X_tr_w.shape, F_tr_w.shape, y_tr)

    ds_tr = tf.data.Dataset.from_tensor_slices(((X_tr_w, F_tr_w), y_tr_w)) \
        .shuffle(len(y_tr_w)) \
        .batch(64) \
        .prefetch(tf.data.AUTOTUNE)

    ds_te = tf.data.Dataset.from_tensor_slices(((X_te_w,), y_te_w)) \
        .batch(64) \
        .prefetch(tf.data.AUTOTUNE)
else:
    X_tr_w, y_tr_w = build_windowed_xy(meta_tr[fold][mask_tr], dict_blocks, win_len, stride)
    X_te_w, y_te_w = build_windowed_xy(meta_te[fold][mask_te], dict_blocks, win_len, stride)

    print(X_tr_w.shape, y_tr_w.shape)  # z.B# . (N, 256, 4), (N,)

    ds_tr = tf.data.Dataset.from_tensor_slices((X_tr_w, y_tr_w)) \
        .shuffle(len(y_tr_w)) \
        .batch(64) \
        .prefetch(tf.data.AUTOTUNE)

    ds_te = tf.data.Dataset.from_tensor_slices((X_te_w, y_te_w)) \
        .batch(64) \
        .prefetch(tf.data.AUTOTUNE)

# Building a feature Table
Building a feature table to feed it into the DNN. To see if it helps

In [ ]:
from auswertung_features.parameterraum import DataConfig
from auswertung_features.build_features import build_feature_table
from auswertung_features.parameter_suche import data_cfg_key
import os

def tsfel_feature(win: int=0, step: int=50, feature_set_name="default_tsfel"):
    tsfel_cfg = tsfel.get_features_by_domain()#json_path="../auswertung_features/tsfel_conf.json")
    dcfg = DataConfig(
        trim=0,
        min_len=win,
        win=win, # feeding the feature table the whole sequence if win = 0. To feed sth different win should be the same as min_len. TSFEL can't work with different win lengths
        step=step, # arbitrary value. It should be the same as stride but if we take the whole sequence, stride may be not defined even though we wouldn't use it
        feature_set_name= feature_set_name
    )
    feature_cache: dict[str, pd.DataFrame] = {}

    k = data_cfg_key(dcfg)
    if k not in feature_cache:
        if os.path.isfile(f"../auswertung_features/cache/feature_{k}.csv"):
            feature_cache[k] = pd.read_csv(f"../auswertung_features/cache/feature_{k}.csv", index_col = 0, dtype={"subject_id":str})
        else:
            feature_cache[k] = build_feature_table(meta=meta, data_cfg=dcfg, tsfel_cfg=tsfel_cfg)
            feature_cache[k].to_csv(f"../auswertung_features/cache/feature_{k}.csv")

    #feature_tensor = tf.convert_to_tensor(feature_cache[k].iloc[:,6:])
    #return feature_tensor
    return feature_cache[k]

# First Test of CNN and Gru
Down here is a first idea or test of the 1d CNN. If you input the whole sequence as train set, you get a max accuracy on the validation set on around 0.5/0.55


In [ ]:
import keras
initializer = tf.keras.initializers.HeNormal(seed = 0)

wd = 1e-4

inputs = layers.Input(shape=(None, 4))
x = layers.Conv1D(8, 7, padding="same", activation="leaky_relu", kernel_initializer=initializer,
                  kernel_regularizer=keras.regularizers.l2(wd))(inputs)
#x = layers.MaxPooling1D()(x)
x = layers.SpatialDropout1D(0.2)(x)
x = layers.Conv1D(16, 5, padding="same", activation="leaky_relu", kernel_regularizer=keras.regularizers.l2(wd))(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(5, activation="softmax", kernel_regularizer=keras.regularizers.l2(wd))(x)

model = models.Model(inputs, outputs)
optimizer = tf.keras.optimizers.Adadelta(learning_rate=1.0)
loss = keras.losses.SparseCategoricalCrossentropy()
model.compile(optimizer=optimizer,
              loss=loss,
              metrics=["accuracy"],)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(model.summary())

model.fit(ds_tr, validation_data=ds_te, epochs=1000, callbacks=[tensorboard_callback])



This one is the first test to implement a 1d CNN --> GRU (RNN). It is pretty bad

In [ ]:
initializer = tf.keras.initializers.HeNormal(seed = 0)

inputs = layers.Input(shape=(None, 4))
x = layers.Conv1D(32, 7, padding="same", activation="leaky_relu",
                  kernel_initializer=initializer)(inputs)
x = layers.MaxPooling1D()(x)
x = layers.GRU(16, activation="leaky_relu")(x)
outputs = layers.Dense(5, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"],)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(model.summary())

model.fit(ds_tr, validation_data=ds_te, epochs=1000, callbacks=[tensorboard_callback])


Building a GRU Model which works with 2 inputs

In [77]:
def build_GRU_model(F=624, wd=1e-3, initializer="he_normal", lr=1e-3):
    init = tf.keras.initializers.get(initializer)

    ts_inputs = tf.keras.layers.Input(shape=(None, 4), name="ts")
    x = tf.keras.layers.GRU(
        8, kernel_initializer=init,
        kernel_regularizer=tf.keras.regularizers.l2(wd),
        activation="leaky_relu"
    )(ts_inputs)
    x = tf.keras.layers.Dropout(0.5)(x)

    feat_in = tf.keras.layers.Input(shape=(F,), name="feat")
    f = tf.keras.layers.LayerNormalization()(feat_in)
    f = tf.keras.layers.Dropout(0.3)(f)
    f = tf.keras.layers.Dense(
        128, kernel_initializer=init,
        kernel_regularizer=tf.keras.regularizers.l2(wd),
        activation="leaky_relu"
    )(f)
    f = tf.keras.layers.Dropout(0.5)(f)
    f = tf.keras.layers.Dense(
        64, kernel_initializer=init,
        kernel_regularizer=tf.keras.regularizers.l2(wd),
        activation="leaky_relu"
    )(f)

    h = tf.keras.layers.Concatenate()([x, f])
    h = tf.keras.layers.Dense(
        16, kernel_initializer=init,
        kernel_regularizer=tf.keras.regularizers.l2(wd),
        activation="leaky_relu"
    )(h)
    h = tf.keras.layers.Dropout(0.3)(h)
    outputs = tf.keras.layers.Dense(5, activation="softmax")(h)

    model = tf.keras.Model(inputs=[ts_inputs, feat_in], outputs=outputs)

    loss = tf.keras.losses.SparseCategoricalCrossentropy()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=loss,
        metrics=["accuracy", tf.keras.metrics.SparseCategoricalCrossentropy(name="ce")]
    )
    return model

model = build_GRU_model()

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(model.summary())

model.fit(ds_tr, validation_data=ds_te, epochs=2000, callbacks=[tensorboard_callback])

Model: "functional_22"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ feat (InputLayer)   │ (None, 624)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 624)       │      1,248 │ feat[0][0]        │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_74          │ (None, 624)       │          0 │ layer_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ts (InputLayer)     │ (None, None, 4)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_78 (Dense)    │ (None, 128)       │     80,000 │ dropout_74[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_5 (GRU)         │ (None, 8)         │        336 │ ts[0][0]          │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_75          │ (None, 128)       │          0 │ dense_78[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_73          │ (None, 8)         │          0 │ gru_5[0][0]       │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_79 (Dense)    │ (None, 64)        │      8,256 │ dropout_75[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_21      │ (None, 72)        │          0 │ dropout_73[0][0], │
│ (Concatenate)       │                   │            │ dense_79[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_80 (Dense)    │ (None, 16)        │      1,168 │ concatenate_21[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_76          │ (None, 16)        │          0 │ dense_80[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_81 (Dense)    │ (None, 5)         │         85 │ dropout_76[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 91,093 (355.83 KB)

 Trainable params: 91,093 (355.83 KB)

 Non-trainable params: 0 (0.00 B)

...

/home/manhl/anaconda3/envs/bpa-ba/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


     34/Unknown 5s 114ms/step - accuracy: 0.2453 - ce: 10.5821 - loss: 11.0347

/home/manhl/anaconda3/envs/bpa-ba/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


34/34 ━━━━━━━━━━━━━━━━━━━━ 5s 122ms/step - accuracy: 0.2453 - ce: 10.5821 - loss: 11.0347 - val_accuracy: 0.3474 - val_ce: 5.8865 - val_loss: 6.3282
Epoch 2/2000
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 119ms/step - accuracy: 0.2585 - ce: 6.8946 - loss: 7.3254 - val_accuracy: 0.4316 - val_ce: 5.0071 - val_loss: 5.4263
Epoch 3/2000
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 119ms/step - accuracy: 0.2764 - ce: 5.6909 - loss: 6.0999 - val_accuracy: 0.4737 - val_ce: 3.7706 - val_loss: 4.1687
Epoch 4/2000
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 116ms/step - accuracy: 0.3009 - ce: 4.6536 - loss: 5.0414 - val_accuracy: 0.5158 - val_ce: 3.2073 - val_loss: 3.5835
Epoch 5/2000
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step - accuracy: 0.3085 - ce: 4.9020 - loss: 5.2685 - val_accuracy: 0.5474 - val_ce: 2.7027 - val_loss: 3.0592
Epoch 6/2000
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 117ms/step - accuracy: 0.3236 - ce: 3.7360 - loss: 4.0838 - val_accuracy: 0.5368 - val_ce: 2.7149 - val_loss: 3.0534
Epoch 7/2000
34/34 ━━━━━━━━━━━━━━━━━━━━ 4s 120ms/st

KeyboardInterrupt: 

Trying a standard model which was described by Kiranyaz et al. (2021) in 1D convolutional neural networks and applications: A survey

In [ ]:
import keras
initializer = tf.keras.initializers.HeNormal(seed = 0)
inputs = layers.Input(shape=(None, 4))

x = layers.Conv1D(20, 15, padding="same", activation="leaky_relu", kernel_initializer=initializer)(inputs)
x = layers.Conv1D(40, 24, padding="same", activation="leaky_relu")(x)
x = layers.Conv1D(60, 30, padding="same", activation="leaky_relu")(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(10, activation="softmax")(x)
outputs = layers.Dense(5, activation="softmax")(x)

model = models.Model(inputs, outputs)
loss = keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adadelta(learning_rate=1.0)
model.compile(optimizer=optimizer,
              loss=loss,
              metrics=["accuracy"],)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(model.summary())

model.fit(ds_tr, validation_data=ds_te, epochs=2000, callbacks=[tensorboard_callback])

## Trying a multi-input
This will use the models from before but this time with features

In [ ]:
import keras
initializer = tf.keras.initializers.HeNormal(seed = 0)

wd_ts = 1e-4
wd_feat = 1e-2
wd_head = 1e-3
F = 624 # number of used features

# input 1 time series (T,4)
ts_inputs = layers.Input(shape=(None, 4), name = "ts")
x = layers.Conv1D(8, 7, padding="same", activation="leaky_relu", kernel_initializer=initializer,
                  kernel_regularizer=keras.regularizers.l2(wd_ts))(ts_inputs)
#x = layers.MaxPooling1D()(x)
x = layers.SpatialDropout1D(0.2)(x)
x = layers.Conv1D(16, 5, padding="same", activation="relu", kernel_regularizer=keras.regularizers.l2(wd_ts))(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.5)(x)

# input 2 features (n_feat,)
feat_in = layers.Input(shape=(F,), name="feat")
f = layers.LayerNormalization()(feat_in)
f = layers.Dropout(0.3)(f)
f = layers.Dense(128, kernel_initializer=initializer, kernel_regularizer=keras.regularizers.l2(wd_feat), activation="leaky_relu")(f)
f = layers.Dropout(0.5)(f)
f = layers.Dense(64, kernel_initializer=initializer, kernel_regularizer=keras.regularizers.l2(wd_feat), activation="leaky_relu")(f)

# Combine
h = layers.Concatenate()([x, f])

h = layers.Dense(
    128,
    kernel_initializer=initializer,
    kernel_regularizer=keras.regularizers.l2(wd_head),
    activation="leaky_relu"
)(h)
h = layers.Dropout(0.3)(h)
outputs = layers.Dense(5, activation="softmax")(h)

model = models.Model(inputs=[ts_inputs, feat_in], outputs=outputs)
optimizer = tf.keras.optimizers.Adadelta(learning_rate=1.0)
loss = keras.losses.SparseCategoricalCrossentropy(name="loss")
model.compile(optimizer=optimizer,
              loss=loss,
              metrics=["accuracy", loss],)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(model.summary())

model.fit(ds_tr, validation_data=ds_te, epochs=210, callbacks=[tensorboard_callback])

train_eval = model.evaluate(ds_tr, verbose=0)
val_eval   = model.evaluate(ds_te, verbose=0)
print(train_eval, val_eval)



# PET Infrastructure build for classification
Right now it is an AI draft based on Lin et al. on A parallel and efficient transformer deep learning network for continuous estimation of hand kinematics from electromyographic signals

In [ ]:
import tensorflow as tf

class ExternalAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, s=64, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.s = s
        self.dropout = dropout

        self.wk = tf.keras.layers.Dense(s, use_bias=False)       # d -> S
        self.wv = tf.keras.layers.Dense(d_model, use_bias=False) # S -> d
        self.drop = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        # x: (B,T,d)
        a = self.wk(x)                            # (B,T,S)
        a = tf.nn.softmax(a, axis=-1)             # softmax over S (memory slots)
        a = self.drop(a, training=training)
        y = self.wv(a)                            # (B,T,d)
        return y


In [ ]:
class PETBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, s=64, ffn_mult=2, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.ea = ExternalAttention(d_model, s=s, dropout=dropout)

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_model * ffn_mult, activation="leaky_relu"),
            tf.keras.layers.Dropout(dropout),
            tf.keras.layers.Dense(d_model),
        ])
        self.drop = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        # Pre-Norm
        h = self.norm(x)

        # parallel branches
        attn_out = self.ea(h, training=training)
        ffn_out  = self.ffn(h, training=training)

        y = attn_out + ffn_out
        y = self.drop(y, training=training)

        # residual
        return x + y


In [ ]:
def build_pet_lite_classifier(
    win_len,
    n_ch=4,
    n_classes=5,
    d_model=64,
    n_blocks=2,
    s=64,
    dropout=0.2,
    conv_stride=2
):
    inp = tf.keras.Input(shape=(win_len, n_ch), name="emg")

    # Optional: Normalization layer (adapt() separat auf Train-Dataset!)
    norm = tf.keras.layers.Normalization(axis=-1)
    x = norm(inp)
    #x = inp

    # Conv frontend (patch/downsample): T -> ceil(T/stride)
    x = tf.keras.layers.Conv1D(d_model, kernel_size=9, strides=conv_stride,
                               padding="same", activation="relu")(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    # PET blocks
    for i in range(n_blocks):
        x = PETBlock(d_model=d_model, s=s, ffn_mult=2, dropout=dropout, name=f"pet_{i}")(x)

    # Pooling over time (sequence-to-vector)
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    out = tf.keras.layers.Dense(n_classes, name="logits"<)(x)

    model = tf.keras.Model(inp, out, name="pet_lite_emg")
    return model


In [ ]:
model = build_pet_lite_classifier(win_len=win_len, n_ch=4, n_classes=5,
                                  d_model=32, n_blocks=2, s=32, dropout=0.5)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

print(model.summary())

model.fit(ds_tr, validation_data=ds_te, epochs=1000, callbacks=[tensorboard_callback])